In [13]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
import matplotlib

In [14]:
aberporth = np.load("aberporth.npy")
cardiff = np.load("cardiff.npy")
valley = np.load("valley.npy")

print(aberporth.shape)
print(cardiff.shape)
print(valley.shape)

print(aberporth)
print(cardiff)
print(valley)

(811, 6)
(556, 6)
(1123, 6)
[[1.957e+03 1.000e+00 8.600e+00 3.900e+00 2.000e+00 8.060e+01]
 [1.957e+03 2.000e+00 8.500e+00 3.200e+00 2.000e+00 8.510e+01]
 [1.957e+03 3.000e+00 1.220e+01 7.400e+00 0.000e+00 8.310e+01]
 ...
 [2.024e+03 5.000e+00 1.550e+01 9.800e+00 0.000e+00 4.820e+01]
 [2.024e+03 6.000e+00 1.550e+01 1.030e+01 0.000e+00 3.360e+01]
 [2.024e+03 7.000e+00 1.740e+01 1.180e+01 0.000e+00 6.900e+01]]
[[1977.     9.    18.3    8.1    0.    48.6]
 [1977.    10.    15.4    9.     0.    76. ]
 [1977.    11.    10.3    4.7    4.   147.9]
 ...
 [2023.    10.    17.4    9.6    0.   178.4]
 [2023.    11.    12.2    5.1    4.   202. ]
 [2023.    12.    10.9    5.7    4.   158.8]]
[[1.930e+03 1.200e+01 8.600e+00 5.500e+00 0.000e+00 1.303e+02]
 [1.931e+03 1.000e+00 8.000e+00 4.200e+00 0.000e+00 6.620e+01]
 [1.931e+03 2.000e+00 7.600e+00 3.700e+00 0.000e+00 6.060e+01]
 ...
 [2.024e+03 5.000e+00 1.700e+01 1.050e+01 0.000e+00 4.480e+01]
 [2.024e+03 6.000e+00 1.680e+01 1.080e+01 0.000e+00 3.0

I will select 3 models to compare. Random Forests, SVM and kNN. I predict random forests will work the best. I picked these 3 because I think they are going to work best with the low number of features - 5. Having random forest also allows us to see how a network technique compares against the other two, and kNN allows us to see how well a computationally cheaper algorithm will compare. SVM is kind of the jack of all trades here I guess. 

In [15]:
#Preparing data
#split into X and Y
aberporthX = aberporth[:, :5]
aberporthY = aberporth[:, -1]

cardiffX = cardiff[:, :5]
cardiffY = cardiff[:, -1]

valleyX = valley[:, :5]
valleyY = valley[:, -1]

#Training and testing datasets
aTrainX, aTestX, aTrainY, aTestY = train_test_split(aberporthX, aberporthY, test_size=0.3)
cTrainX, cTestX, cTrainY, cTestY = train_test_split(cardiffX, cardiffY, test_size=0.3)
vTrainX, vTestX, vTrainY, vTestY = train_test_split(valleyX, valleyY, test_size=0.3)

#Scaling data
#KNN and SVM require data scaling since the method of regression behind them rely on distance between data points
#Therefore we will create scaled datasets for these models so that no single feature dominates the decision making
scaler = StandardScaler()
aTrainXScaled = scaler.fit_transform(aTrainX)
aTestXScaled = scaler.transform(aTestX)
cTrainXScaled = scaler.fit_transform(cTrainX)
cTestXScaled = scaler.transform(cTestX)
vTrainXScaled = scaler.fit_transform(vTrainX)
vTestXScaled = scaler.transform(vTestX)

In [16]:
#KNN regressor
paramGridKnn = {
    "n_neighbors": range(1,51),
    "weights": ['uniform', 'distance'],
}
gridSearchKnn = GridSearchCV(estimator=KNeighborsRegressor(), param_grid=paramGridKnn, cv=5, n_jobs=-1)

#Training for Aberporth
gridSearchKnn.fit(aTrainXScaled, aTrainY)
Aknn = KNeighborsRegressor()
Aknn.set_params(**gridSearchKnn.best_params_)

#Training for Cardiff
gridSearchKnn.fit(cTrainXScaled, cTrainY)
Cknn = KNeighborsRegressor()
Cknn.set_params(**gridSearchKnn.best_params_)

#Training for Valley
gridSearchKnn.fit(vTrainXScaled, vTrainY)
Vknn = KNeighborsRegressor()
Vknn.set_params(**gridSearchKnn.best_params_)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",46
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None


In [18]:
#Random Forest regressor
paramGridRf = {
    "n_estimators": [50,500],
    "criterion": ['squared_error', 'absolute_error', 'poisson'],
    "max_features": ['sqrt', 'log2']
}
gridSearchRf = GridSearchCV(estimator=RandomForestRegressor(), param_grid=paramGridRf, cv=5, n_jobs=-1)

#Training for Aberporth
gridSearchRf.fit(aTrainX, aTrainY)
aRf = RandomForestRegressor()
aRf.set_params(**gridSearchRf.best_params_)

#Training for Cardiff
gridSearchRf.fit(cTrainX, cTrainY)
cRf = RandomForestRegressor()
cRf.set_params(**gridSearchRf.best_params_)

#Training for Valley
gridSearchRf.fit(vTrainX, vTrainY)
vRf = RandomForestRegressor()
vRf.set_params(**gridSearchRf.best_params_)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'absolute_error'
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the 

In [21]:
#Support Vector Machine Regressor
paramGridSvm = {
    "kernel": ['rbf', 'linear', 'poly', 'sigmoid'],
    "C": [0.01, 10],
    "epsilon": [0.01, 10]
}
gridSearchSvm = GridSearchCV(estimator=SVR(), param_grid=paramGridSvm, cv=5, n_jobs=-1)

#Training for Aberporth
gridSearchSvm.fit(aTrainXScaled, aTrainY)
aSvr = SVR()
aSvr.set_params(**gridSearchSvm.best_params_)

#Training for Cardiff
gridSearchSvm.fit(cTrainXScaled, cTrainY)
cSvr = SVR()
cSvr.set_params(**gridSearchSvm.best_params_)

#Training for Valley
gridSearchSvm.fit(vTrainXScaled, vTrainY)
vSvr = SVR()
vSvr.set_params(**gridSearchSvm.best_params_)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.01
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1
